# Chapter 74: Security Orchestration & SOAR

**Volume 4 — Security Operations**

**A skilled analyst takes 35 minutes on a phishing alert. A SOAR playbook takes 90 seconds.**
The difference is not intelligence — it's mechanical context gathering replaced by automation.
SOAR doesn't replace analyst judgment. It reserves judgment for decisions that need it.

### What You Will Build — 5 Demos

| Demo | Pattern | What It Demonstrates |
|------|---------|---------------------|
| **1. Alert Enrichment** | Pipeline/chaining | Receive alert → query threat intel → add context → score |
| **2. Playbook Router** | LLM classification | Classify alert type → select correct playbook → initialize |
| **3. Multi-Tool Orchestration** | Parallel execution | EDR + Firewall + Ticketing APIs called in sequence |
| **4. Case Management** | State machine | Create/update/close incident cases, track state |
| **5. Full SOAR Pipeline** | CWD model | Alert → Enrich → Route → Execute → Case update → Notify |

### The Core Insight
> **Sequential investigation: 35 minutes. Parallel SOAR pipeline: 90 seconds.**
> The math: total time = SUM(steps) for manual work, MAX(steps) for parallel automation.
> SOAR converts the sum into the max — and runs identically at 4:30pm Friday as at 9am Monday.
> Consistency is not just efficiency. It is quality.

In [ ]:
# pip install anthropic
import os, json, re, math, time, random
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from collections import defaultdict

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

api_key = os.environ.get('ANTHROPIC_API_KEY', '')
USE_API = bool(api_key)

if USE_API:
    from anthropic import Anthropic
    client = Anthropic()
    print("LLM analyst: ACTIVE (Anthropic API)")
else:
    print("LLM analyst: SIMULATION MODE (set ANTHROPIC_API_KEY for real analysis)")

def llm_analyze(prompt: str, max_tokens: int = 200) -> str:
    if USE_API:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.content[0].text.strip()
    return "Simulation: " + prompt[:80] + "..."

print("Setup complete.")

## Demo 1: Alert Enrichment Workflow

**The pipeline pattern** decomposes a complex investigation into a sequence of simpler steps.
Each step's output becomes the next step's input — like a packet flowing through a processing chain.

A raw alert arrives with minimal context: an IP, a timestamp, an alert type.
Enrichment adds the context an analyst needs to make a decision:

| Enrichment Step | Data Source | What It Adds |
|----------------|-------------|-------------|
| **GeoIP lookup** | MaxMind / ip-api | Country, ASN, hosting provider |
| **Threat Intel** | VirusTotal / Shodan | Malicious verdicts, known IOCs |
| **Asset lookup** | CMDB / ServiceNow | Is the target host critical? Who owns it? |
| **History check** | SIEM / local cache | Has this IP appeared before? |
| **Risk score** | Weighted formula | Combined score driving playbook routing |

```
Raw alert: {src_ip: 185.220.101.47, type: port_scan, target: 10.0.5.44}
    |
    v
GeoIP: Russia / Tor exit node / AS208323
    |
    v
Threat Intel: 47 malicious verdicts (VirusTotal), 9.2/10 Shodan risk
    |
    v
Asset: 10.0.5.44 = payment-processor-01, CRITICALITY=CRITICAL
    |
    v
Risk Score: 0.93 -> Route to CRITICAL playbook
```

*Network analogy: alert enrichment is your BGP RIB. The raw route announcement is
one data point. The enriched RIB entry adds AS path, community tags, local preference,
and MED — all the context needed to make the right forwarding decision.*

In [ ]:
# ── Demo 1: Alert Enrichment Workflow ─────────────────────────────────────────

@dataclass
class Alert:
    """Raw alert as it arrives from the SIEM or detection tool."""
    alert_id: str
    alert_type: str        # port_scan / phishing / malware / auth_anomaly
    src_ip: str
    target_host: str
    timestamp: str
    raw_details: str

@dataclass
class EnrichedAlert:
    """Alert with all enrichment context attached."""
    alert: Alert
    geo: Dict           = field(default_factory=dict)
    threat_intel: Dict  = field(default_factory=dict)
    asset_info: Dict    = field(default_factory=dict)
    history: Dict       = field(default_factory=dict)
    risk_score: float   = 0.0
    llm_summary: str    = ""

# ── Simulated enrichment functions (replace with real API calls in production) ──

def enrich_geo(ip: str) -> Dict:
    """Simulate GeoIP lookup (in production: MaxMind GeoIP2 or ip-api.com)."""
    geo_db = {
        "185.220.101.47": {"country": "Russia", "asn": "AS208323",
                            "org": "Tor exit node", "risk": "high"},
        "45.33.32.156":   {"country": "United States", "asn": "AS63949",
                            "org": "Linode/Akamai", "risk": "medium"},
        "8.8.8.8":        {"country": "United States", "asn": "AS15169",
                            "org": "Google LLC", "risk": "low"},
    }
    return geo_db.get(ip, {"country": "Unknown", "asn": "Unknown",
                           "org": "Unknown", "risk": "medium"})

def enrich_threat_intel(ip: str) -> Dict:
    """Simulate VirusTotal / threat intel lookup."""
    intel_db = {
        "185.220.101.47": {"vt_malicious": 47, "vt_total": 94,
                            "tags": ["tor", "scanner", "brute-force"],
                            "first_seen": "2022-03-15", "last_seen": "2024-12-01"},
        "45.33.32.156":   {"vt_malicious": 3, "vt_total": 94,
                            "tags": ["vps", "possibly-malicious"],
                            "first_seen": "2023-08-01", "last_seen": "2024-11-20"},
        "8.8.8.8":        {"vt_malicious": 0, "vt_total": 94,
                            "tags": ["dns", "google"], "first_seen": "2010-01-01"},
    }
    return intel_db.get(ip, {"vt_malicious": 0, "vt_total": 94, "tags": []})

def enrich_asset(host: str) -> Dict:
    """Simulate CMDB / asset inventory lookup."""
    cmdb = {
        "10.0.5.44":  {"hostname": "payment-processor-01", "owner": "finance-team",
                       "criticality": "CRITICAL", "handles_pci": True,
                       "os": "Ubuntu 22.04", "patch_level": "current"},
        "10.0.1.50":  {"hostname": "dev-workstation-12", "owner": "alice@corp.com",
                       "criticality": "LOW", "handles_pci": False, "os": "Windows 11"},
        "10.0.2.100": {"hostname": "web-proxy-01", "owner": "network-team",
                       "criticality": "HIGH", "handles_pci": False, "os": "CentOS 8"},
    }
    return cmdb.get(host, {"hostname": host, "criticality": "UNKNOWN", "handles_pci": False})

def enrich_history(ip: str) -> Dict:
    """Simulate SIEM historical lookup — has this IP appeared before?"""
    history_db = {
        "185.220.101.47": {"prior_alerts": 12, "first_alert": "2024-09-01",
                            "alert_types": ["port_scan", "brute_force"],
                            "previously_blocked": True},
        "45.33.32.156":   {"prior_alerts": 1, "first_alert": "2024-11-15",
                            "alert_types": ["port_scan"], "previously_blocked": False},
        "8.8.8.8":        {"prior_alerts": 0, "previously_blocked": False},
    }
    return history_db.get(ip, {"prior_alerts": 0, "previously_blocked": False})

def compute_risk_score(enriched: EnrichedAlert) -> float:
    """
    Weighted risk score from enrichment context.
    Formula: 0.35*threat_intel + 0.25*geo_risk + 0.25*asset_crit + 0.15*history
    """
    # Threat intel score (0-1)
    vt_mal = enriched.threat_intel.get("vt_malicious", 0)
    vt_tot = enriched.threat_intel.get("vt_total", 94)
    ti_score = vt_mal / vt_tot if vt_tot > 0 else 0.0

    # Geo risk score (0-1)
    geo_map = {"high": 1.0, "medium": 0.5, "low": 0.1}
    geo_score = geo_map.get(enriched.geo.get("risk", "medium"), 0.5)

    # Asset criticality score (0-1)
    crit_map = {"CRITICAL": 1.0, "HIGH": 0.75, "MEDIUM": 0.5, "LOW": 0.2, "UNKNOWN": 0.4}
    asset_score = crit_map.get(enriched.asset_info.get("criticality", "UNKNOWN"), 0.4)
    if enriched.asset_info.get("handles_pci"):
        asset_score = min(asset_score + 0.2, 1.0)  # PCI systems get bonus risk

    # History score (0-1)
    prior = enriched.history.get("prior_alerts", 0)
    hist_score = min(prior / 10.0, 1.0)
    if enriched.history.get("previously_blocked"):
        hist_score = min(hist_score + 0.3, 1.0)

    combined = (ti_score * 0.35 + geo_score * 0.25 +
                asset_score * 0.25 + hist_score * 0.15)
    return round(combined, 3)

def enrich_alert(alert: Alert) -> EnrichedAlert:
    """Full enrichment pipeline — all context stages applied in sequence."""
    enriched = EnrichedAlert(alert=alert)

    print(f"[ENRICH] Alert {alert.alert_id}: {alert.alert_type} | {alert.src_ip} -> {alert.target_host}")

    # Pipeline: each stage adds context
    enriched.geo          = enrich_geo(alert.src_ip)
    print(f"  [1/4] GeoIP: {enriched.geo['country']} / {enriched.geo['org']} / risk={enriched.geo['risk']}")

    enriched.threat_intel = enrich_threat_intel(alert.src_ip)
    print(f"  [2/4] ThreatIntel: {enriched.threat_intel.get('vt_malicious',0)}/"
          f"{enriched.threat_intel.get('vt_total',94)} VT malicious | "
          f"tags={enriched.threat_intel.get('tags',[])}")

    enriched.asset_info   = enrich_asset(alert.target_host)
    print(f"  [3/4] Asset: {enriched.asset_info.get('hostname')} | "
          f"criticality={enriched.asset_info.get('criticality')} | "
          f"PCI={enriched.asset_info.get('handles_pci')}")

    enriched.history      = enrich_history(alert.src_ip)
    print(f"  [4/4] History: {enriched.history.get('prior_alerts',0)} prior alerts | "
          f"blocked_before={enriched.history.get('previously_blocked')}")

    enriched.risk_score   = compute_risk_score(enriched)

    # LLM summary of the enriched context
    summary_prompt = (
        f"Summarize this enriched security alert for a SOC analyst:\n"
        f"Alert: {alert.alert_type} from {alert.src_ip} targeting {alert.target_host}\n"
        f"GeoIP: {enriched.geo}\n"
        f"Threat Intel: {enriched.threat_intel}\n"
        f"Asset: {enriched.asset_info}\n"
        f"History: {enriched.history}\n"
        f"Risk Score: {enriched.risk_score}\n"
        f"In 2 sentences: what is the risk and what is the most important factor? Be specific."
    )
    enriched.llm_summary = llm_analyze(summary_prompt, max_tokens=120)

    sev = "CRITICAL" if enriched.risk_score >= 0.8 else \
          "HIGH"     if enriched.risk_score >= 0.6 else \
          "MEDIUM"   if enriched.risk_score >= 0.35 else "LOW"

    print(f"\n  [RESULT] Risk Score: {enriched.risk_score:.3f} -> {sev}")
    print(f"  [LLM]   {enriched.llm_summary}")
    return enriched

# ── Run enrichment on two alerts ───────────────────────────────────────────────
print("=" * 65)
print("ALERT ENRICHMENT PIPELINE")
print("=" * 65)

alerts = [
    Alert("ALT-001", "port_scan", "185.220.101.47", "10.0.5.44",
          "2024-12-01T03:14:00Z",
          "SYN scan from external IP targeting payment processor on ports 22,80,443,3389"),
    Alert("ALT-002", "port_scan", "45.33.32.156", "10.0.1.50",
          "2024-12-01T09:22:00Z",
          "SYN scan targeting developer workstation on port 22"),
]

enriched_alerts = []
for alert in alerts:
    print()
    enriched = enrich_alert(alert)
    enriched_alerts.append(enriched)
    print("-" * 65)

print(f"\n[PIPELINE] {len(enriched_alerts)} alerts enriched. Highest risk: "
      f"{max(e.risk_score for e in enriched_alerts):.3f}")

## Demo 2: Playbook Router

**The routing pattern** uses an LLM classifier to select the correct investigation playbook
from a library of options — without requiring rigid rule trees.

A rule-based router looks like:
```
if alert_type == "phishing": -> phishing_playbook
if alert_type == "malware":  -> malware_playbook
```

This fails when context matters more than type. A "privilege escalation" alert at 3am
on a critical server by a never-seen-before account is not the same as the same alert
type at 2pm by a known admin doing routine work.

**LLM routing** evaluates the full context holistically:

| Playbook | Triggers On |
|----------|-------------|
| `P01_PHISHING_INVESTIGATION` | Email-delivered threats, BEC, spearphishing |
| `P02_MALWARE_CONTAINMENT` | Endpoint IOCs, process execution, file drops |
| `P03_ACCOUNT_COMPROMISE` | Auth anomalies, impossible travel, credential stuffing |
| `P04_NETWORK_INTRUSION` | Port scans, lateral movement, C2 beacons |
| `P05_DATA_EXFILTRATION` | Large outbound transfers, DLP triggers, staging |
| `P06_INSIDER_THREAT` | Privileged user anomalies, policy violations |
| `P07_CRITICAL_INCIDENT` | Multi-stage, multi-system, high-impact scenarios |

*Network analogy: the playbook router is your BGP policy router. It doesn't just
look at the destination prefix — it evaluates AS path, community tags, local pref,
and MED before selecting the next hop. Context-aware routing, not just type matching.*

In [ ]:
# ── Demo 2: Playbook Router ────────────────────────────────────────────────────

PLAYBOOK_CATALOG = {
    "P01_PHISHING_INVESTIGATION": {
        "description": "Email-delivered threats: BEC, spearphishing, malicious attachments",
        "steps": ["email_header_analysis", "url_detonation", "sender_reputation",
                  "recipient_risk_assessment", "quarantine_decision"],
        "avg_duration_min": 3,
        "auto_actions": ["quarantine_email", "extract_iocs"],
    },
    "P02_MALWARE_CONTAINMENT": {
        "description": "Endpoint IOCs: process execution, file drops, registry changes",
        "steps": ["process_tree_analysis", "file_hash_lookup", "network_connections",
                  "memory_scan", "isolation_decision"],
        "avg_duration_min": 8,
        "auto_actions": ["snapshot_endpoint", "pull_process_list"],
    },
    "P03_ACCOUNT_COMPROMISE": {
        "description": "Auth anomalies: impossible travel, cred stuffing, MFA bypass",
        "steps": ["auth_log_review", "geo_velocity_check", "device_fingerprint",
                  "session_inventory", "account_action_decision"],
        "avg_duration_min": 5,
        "auto_actions": ["flag_session", "step_up_mfa"],
    },
    "P04_NETWORK_INTRUSION": {
        "description": "External attack: port scans, lateral movement, C2 beacons",
        "steps": ["ip_reputation", "port_scan_analysis", "lateral_movement_check",
                  "firewall_log_review", "block_decision"],
        "avg_duration_min": 4,
        "auto_actions": ["pull_firewall_logs", "check_ids_alerts"],
    },
    "P05_DATA_EXFILTRATION": {
        "description": "Data loss: large outbound transfers, DLP triggers, staging",
        "steps": ["data_volume_analysis", "destination_reputation", "user_context",
                  "dlp_review", "containment_decision"],
        "avg_duration_min": 12,
        "auto_actions": ["capture_netflow", "dlp_hold"],
    },
    "P07_CRITICAL_INCIDENT": {
        "description": "Multi-stage attack: high blast radius, executive notification required",
        "steps": ["immediate_scope_assessment", "parallel_evidence_collection",
                  "stakeholder_notification", "ir_team_activation", "crisis_coordination"],
        "avg_duration_min": 60,
        "auto_actions": ["page_ir_lead", "open_war_room_channel"],
    },
}

def route_to_playbook(enriched: EnrichedAlert) -> Dict:
    """
    Use LLM to select the correct playbook from the catalog.
    Returns: selected playbook ID, confidence, reasoning.
    """
    alert = enriched.alert
    catalog_summary = "\n".join(
        f"  {pid}: {pdata['description']}"
        for pid, pdata in PLAYBOOK_CATALOG.items()
    )

    routing_prompt = (
        f"You are a SOAR playbook router. Select the best playbook for this alert.\n\n"
        f"ALERT:\n"
        f"  Type: {alert.alert_type}\n"
        f"  Source IP: {alert.src_ip}\n"
        f"  Target: {enriched.asset_info.get('hostname', alert.target_host)} "
        f"(criticality={enriched.asset_info.get('criticality')}, "
        f"PCI={enriched.asset_info.get('handles_pci')})\n"
        f"  Risk Score: {enriched.risk_score}\n"
        f"  Threat Intel: {enriched.threat_intel.get('tags', [])}\n"
        f"  History: {enriched.history.get('prior_alerts', 0)} prior alerts, "
        f"previously blocked: {enriched.history.get('previously_blocked')}\n"
        f"  Details: {alert.raw_details}\n\n"
        f"AVAILABLE PLAYBOOKS:\n{catalog_summary}\n\n"
        f"Respond in EXACTLY this format (no other text):\n"
        f"PLAYBOOK: <playbook_id>\n"
        f"CONFIDENCE: <HIGH|MEDIUM|LOW>\n"
        f"REASON: <one sentence why>"
    )

    if USE_API:
        response = llm_analyze(routing_prompt, max_tokens=100)
    else:
        # Simulation: simple rule-based fallback
        if enriched.risk_score >= 0.8:
            response = ("PLAYBOOK: P07_CRITICAL_INCIDENT\n"
                       "CONFIDENCE: HIGH\n"
                       "REASON: High-risk source targeting critical PCI system with prior block history.")
        elif "scan" in alert.alert_type:
            response = ("PLAYBOOK: P04_NETWORK_INTRUSION\n"
                       "CONFIDENCE: HIGH\n"
                       "REASON: External port scan from known malicious IP targeting internal host.")
        else:
            response = ("PLAYBOOK: P04_NETWORK_INTRUSION\n"
                       "CONFIDENCE: MEDIUM\n"
                       "REASON: Network-based alert with external source IP.")

    # Parse the structured response
    result = {"raw": response, "playbook_id": None, "confidence": None, "reason": None}
    for line in response.split("\n"):
        if line.startswith("PLAYBOOK:"):
            result["playbook_id"] = line.split(":", 1)[1].strip()
        elif line.startswith("CONFIDENCE:"):
            result["confidence"] = line.split(":", 1)[1].strip()
        elif line.startswith("REASON:"):
            result["reason"] = line.split(":", 1)[1].strip()

    # Attach playbook details
    pid = result["playbook_id"]
    if pid and pid in PLAYBOOK_CATALOG:
        result["playbook"] = PLAYBOOK_CATALOG[pid]

    return result

# ── Route the enriched alerts ──────────────────────────────────────────────────
print("=" * 65)
print("PLAYBOOK ROUTER — LLM CLASSIFICATION")
print("=" * 65)

routing_decisions = []
for enriched in enriched_alerts:
    print(f"\n[ROUTING] Alert {enriched.alert.alert_id} | Risk={enriched.risk_score}")
    decision = route_to_playbook(enriched)
    routing_decisions.append(decision)

    print(f"  Selected: {decision['playbook_id']}")
    print(f"  Confidence: {decision['confidence']}")
    print(f"  Reason: {decision['reason']}")
    if "playbook" in decision:
        pb = decision["playbook"]
        print(f"  Steps ({len(pb['steps'])}): {' -> '.join(pb['steps'][:3])} ...")
        print(f"  Auto-actions: {pb['auto_actions']}")
        print(f"  Est. duration: {pb['avg_duration_min']} min (automated)")

print(f"\n[ROUTER] {len(routing_decisions)} routing decisions complete.")

## Demo 3: Multi-Tool Orchestration

**The integration hub** is where SOAR gets real power: calling multiple security tools
in sequence (or parallel), each with its own API, authentication, and data format.

A SOAR playbook for a network intrusion might call:

```
1. EDR API     -> check if src_ip was seen on any endpoint in past 24h
2. Firewall API -> verify current ACL state, block src_ip at perimeter
3. SIEM API    -> pull all logs for src_ip in past 7 days
4. Ticketing   -> create incident ticket with all evidence attached
5. Slack/PD    -> notify on-call analyst with priority context
```

Each call uses a different authentication method, returns different data shapes,
and may fail in different ways. The integration hub abstracts this:

| Tool | Auth Method | Rate Limit | Failure Mode |
|------|------------|------------|-------------|
| EDR (CrowdStrike) | OAuth2 bearer | 100 req/min | 429 -> retry after header |
| Firewall (Palo Alto) | API key | 10 req/min | Timeout -> queue |
| SIEM (Splunk) | Session token | Unlimited | Async job -> poll |
| Ticketing (ServiceNow) | Basic auth | 60 req/min | 503 -> retry with backoff |

*Network analogy: multi-tool orchestration is NETCONF over YANG. You speak one
protocol (SOAR action API) and the adapter translates to whatever the device speaks
underneath — Cisco IOS, Juniper JunOS, Palo Alto PAN-OS. MCP does this for AI agents.*

In [ ]:
# ── Demo 3: Multi-Tool Orchestration ──────────────────────────────────────────
import concurrent.futures

@dataclass
class ToolResult:
    """Standardized result from any SOAR integration tool."""
    tool: str
    action: str
    success: bool
    data: Dict
    error: str = ""
    duration_ms: int = 0

def call_edr_api(src_ip: str, target_host: str) -> ToolResult:
    """Simulate CrowdStrike EDR API — check if IP was seen on any endpoint."""
    start = time.time()
    time.sleep(random.uniform(0.05, 0.15))  # simulate API latency

    # Simulated EDR response
    data = {
        "ip_on_endpoint": src_ip in ["185.220.101.47"],
        "endpoint_hits": [
            {"hostname": "workstation-42", "last_seen": "2024-12-01T02:55:00Z",
             "process": "nmap.exe", "user": "SYSTEM"}
        ] if src_ip == "185.220.101.47" else [],
        "quarantine_available": True,
    }
    return ToolResult("edr", "check_ip_presence", True, data,
                      duration_ms=int((time.time()-start)*1000))

def call_firewall_api(src_ip: str, action: str = "check") -> ToolResult:
    """Simulate Palo Alto firewall API — check/modify ACL for source IP."""
    start = time.time()
    time.sleep(random.uniform(0.08, 0.20))  # firewall APIs are slower

    if action == "block":
        data = {
            "action": "blocked",
            "rule_name": f"SOAR-AUTO-BLOCK-{src_ip.replace('.', '-')}",
            "rule_id": f"rule_{random.randint(1000,9999)}",
            "applied_to": ["untrust-to-trust", "untrust-to-dmz"],
            "note": "HUMAN APPROVAL REQUIRED before enforcement",
        }
    else:
        data = {
            "current_rules": "no explicit block",
            "default_action": "deny",
            "existing_permits": [],
        }
    return ToolResult("firewall", action, True, data,
                      duration_ms=int((time.time()-start)*1000))

def call_siem_api(src_ip: str, days: int = 7) -> ToolResult:
    """Simulate Splunk SIEM API — pull all events for source IP."""
    start = time.time()
    time.sleep(random.uniform(0.10, 0.25))  # SIEM searches take time

    data = {
        "total_events": 1247 if src_ip == "185.220.101.47" else 12,
        "event_types": {
            "firewall_deny": 891,
            "port_scan": 312,
            "auth_failure": 44,
        } if src_ip == "185.220.101.47" else {"firewall_deny": 12},
        "first_seen_in_window": f"{days} days ago",
        "target_hosts_attempted": 23 if src_ip == "185.220.101.47" else 1,
        "search_job_id": f"scheduler__{random.randint(100000, 999999)}",
    }
    return ToolResult("siem", "search", True, data,
                      duration_ms=int((time.time()-start)*1000))

def call_ticketing_api(alert: Alert, results: List[ToolResult]) -> ToolResult:
    """Simulate ServiceNow ticketing API — create incident ticket."""
    start = time.time()
    time.sleep(random.uniform(0.05, 0.10))

    evidence_summary = "\n".join(
        f"  - {r.tool.upper()}: {json.dumps(r.data)[:100]}..."
        for r in results if r.success
    )
    ticket_id = f"INC{random.randint(1000000, 9999999)}"
    data = {
        "ticket_id": ticket_id,
        "state": "New",
        "priority": "P1",
        "assignment_group": "SOC-Level-2",
        "description": (
            f"SOAR Auto-Generated Incident\n"
            f"Alert: {alert.alert_type} from {alert.src_ip}\n"
            f"Evidence collected:\n{evidence_summary}"
        ),
        "url": f"https://corp.service-now.com/incident/{ticket_id}",
    }
    return ToolResult("ticketing", "create_incident", True, data,
                      duration_ms=int((time.time()-start)*1000))

def orchestrate_tools(enriched: EnrichedAlert) -> Dict:
    """
    Orchestrate multiple tool calls for a network intrusion response.
    Phase 1: Parallel — EDR, Firewall check, SIEM (no dependencies between them)
    Phase 2: Sequential — Ticketing (needs Phase 1 results)
    """
    alert = enriched.alert
    all_results = []

    print(f"[ORCHESTRATE] Starting tool calls for {alert.alert_id}")
    print(f"  Phase 1: Parallel evidence collection...")

    phase1_start = time.time()
    # Phase 1: Run EDR, Firewall check, and SIEM in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:
        futures = {
            pool.submit(call_edr_api, alert.src_ip, alert.target_host): "edr",
            pool.submit(call_firewall_api, alert.src_ip, "check"): "firewall_check",
            pool.submit(call_siem_api, alert.src_ip, 7): "siem",
        }
        phase1_results = {}
        for future in concurrent.futures.as_completed(futures):
            key = futures[future]
            result = future.result()
            phase1_results[key] = result
            all_results.append(result)

    phase1_duration = time.time() - phase1_start

    for key, r in phase1_results.items():
        status = "OK" if r.success else "FAILED"
        print(f"  [{status}] {r.tool.upper()} ({r.action}): {r.duration_ms}ms")
        if key == "edr" and r.data.get("endpoint_hits"):
            print(f"       EDR MATCH: IP seen on "
                  f"{len(r.data['endpoint_hits'])} endpoint(s)")
        if key == "siem":
            print(f"       SIEM: {r.data.get('total_events')} events, "
                  f"{r.data.get('target_hosts_attempted')} targets attempted")

    print(f"  Phase 1 complete in {phase1_duration*1000:.0f}ms "
          f"(parallel, not {sum(r.duration_ms for r in all_results)}ms sequential)")

    # Phase 2: Create ticket with all phase 1 evidence
    print(f"  Phase 2: Creating incident ticket...")
    ticket_result = call_ticketing_api(alert, all_results)
    all_results.append(ticket_result)
    print(f"  [OK] TICKET created: {ticket_result.data.get('ticket_id')} "
          f"-> {ticket_result.data.get('url')}")

    return {"results": all_results, "ticket": ticket_result.data,
            "phase1_duration_ms": int(phase1_duration*1000)}

# ── Run orchestration on the highest-risk alert ────────────────────────────────
print("=" * 65)
print("MULTI-TOOL ORCHESTRATION")
print("=" * 65)

highest_risk = max(enriched_alerts, key=lambda e: e.risk_score)
orch_results = orchestrate_tools(highest_risk)

print(f"\n[COMPLETE] Ticket: {orch_results['ticket'].get('ticket_id')}")
print(f"[COMPLETE] All evidence collected. Awaiting analyst approval for block action.")

## Demo 4: Case Management State Machine

**A SOAR case** tracks the full lifecycle of a security incident from initial alert
through investigation and response to closure. It's a state machine:

```
NEW -> INVESTIGATING -> PENDING_APPROVAL -> RESPONDING -> CLOSED
         |                                     |
         v                                     v
     ESCALATED                           FALSE_POSITIVE
```

**Valid state transitions:**

| From | To | Trigger |
|------|----|---------|
| `NEW` | `INVESTIGATING` | Alert assigned, evidence collection started |
| `INVESTIGATING` | `PENDING_APPROVAL` | AI decision ready, human gate triggered |
| `INVESTIGATING` | `ESCALATED` | Complexity exceeds automation threshold |
| `PENDING_APPROVAL` | `RESPONDING` | Analyst approved recommended action |
| `PENDING_APPROVAL` | `INVESTIGATING` | Analyst requests more evidence |
| `RESPONDING` | `CLOSED` | Response actions confirmed complete |
| `RESPONDING` | `FALSE_POSITIVE` | Analyst determines no threat after response start |

The state machine prevents invalid transitions — you can't jump from `NEW` to `CLOSED`.

*Network analogy: the case state machine is your STP port state machine.
Blocking -> Listening -> Learning -> Forwarding — no jumping straight to Forwarding.
Each transition has explicit conditions and direction. Invalid transitions are rejected.*

In [ ]:
# ── Demo 4: Case Management State Machine ─────────────────────────────────────
from enum import Enum

class CaseState(str, Enum):
    NEW               = "NEW"
    INVESTIGATING     = "INVESTIGATING"
    PENDING_APPROVAL  = "PENDING_APPROVAL"
    RESPONDING        = "RESPONDING"
    ESCALATED         = "ESCALATED"
    CLOSED            = "CLOSED"
    FALSE_POSITIVE    = "FALSE_POSITIVE"

# Valid transitions: {from_state: [allowed_to_states]}
VALID_TRANSITIONS: Dict[CaseState, List[CaseState]] = {
    CaseState.NEW:              [CaseState.INVESTIGATING, CaseState.FALSE_POSITIVE],
    CaseState.INVESTIGATING:    [CaseState.PENDING_APPROVAL, CaseState.ESCALATED, CaseState.FALSE_POSITIVE],
    CaseState.PENDING_APPROVAL: [CaseState.RESPONDING, CaseState.INVESTIGATING, CaseState.ESCALATED],
    CaseState.RESPONDING:       [CaseState.CLOSED, CaseState.FALSE_POSITIVE, CaseState.ESCALATED],
    CaseState.ESCALATED:        [CaseState.INVESTIGATING, CaseState.RESPONDING, CaseState.CLOSED],
    CaseState.CLOSED:           [],
    CaseState.FALSE_POSITIVE:   [],
}

@dataclass
class CaseEvent:
    """An immutable event in the case timeline."""
    timestamp: str
    actor: str       # SOAR_AUTO / analyst_id
    from_state: CaseState
    to_state: CaseState
    action: str
    details: str

@dataclass
class IncidentCase:
    """A SOAR incident case tracking the full investigation lifecycle."""
    case_id: str
    alert_id: str
    title: str
    state: CaseState = CaseState.NEW
    severity: str = "HIGH"
    assigned_to: str = "SOAR_AUTO"
    timeline: List[CaseEvent] = field(default_factory=list)
    evidence: List[Dict] = field(default_factory=list)
    recommended_action: str = ""
    analyst_decision: str = ""

    def transition(self, new_state: CaseState, actor: str,
                   action: str, details: str) -> bool:
        """Attempt a state transition. Returns True on success."""
        allowed = VALID_TRANSITIONS.get(self.state, [])
        if new_state not in allowed:
            print(f"  [ERROR] Invalid transition: {self.state} -> {new_state}")
            print(f"          Allowed from {self.state}: {[s.value for s in allowed]}")
            return False

        event = CaseEvent(
            timestamp=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            actor=actor,
            from_state=self.state,
            to_state=new_state,
            action=action,
            details=details,
        )
        self.timeline.append(event)
        prev = self.state
        self.state = new_state
        print(f"  [STATE] {prev.value} -> {new_state.value} | {actor}: {action}")
        return True

    def add_evidence(self, source: str, data: Dict):
        """Attach evidence to the case."""
        self.evidence.append({"source": source, "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
                               "data": data})
        print(f"  [EVIDENCE] Added from {source}: {str(data)[:80]}...")

    def generate_summary(self) -> str:
        """Use LLM to generate a case summary for analyst review."""
        evidence_text = json.dumps(self.evidence[:3], indent=2)[:600]
        prompt = (
            f"Generate a concise incident case summary for a SOC analyst:\n\n"
            f"Case: {self.case_id} | {self.title}\n"
            f"Current state: {self.state.value}\n"
            f"Severity: {self.severity}\n"
            f"Evidence collected:\n{evidence_text}\n\n"
            f"Provide: 1) What happened, 2) Why it is concerning, 3) Recommended action. "
            f"Under 100 words. Be specific."
        )
        return llm_analyze(prompt, max_tokens=150)

    def print_timeline(self):
        """Print the full case timeline."""
        print(f"\n  TIMELINE: {self.case_id}")
        for i, ev in enumerate(self.timeline, 1):
            print(f"  {i}. [{ev.timestamp}] {ev.actor}: {ev.action}")
            print(f"     {ev.from_state.value} -> {ev.to_state.value} | {ev.details[:80]}")

# ── Simulate a full case lifecycle ────────────────────────────────────────────
print("=" * 65)
print("CASE MANAGEMENT STATE MACHINE")
print("=" * 65)

case = IncidentCase(
    case_id="CASE-2024-0847",
    alert_id="ALT-001",
    title="External port scan targeting PCI payment processor from known Tor exit node",
    severity="CRITICAL"
)

print(f"\n[CASE] Created: {case.case_id} | State: {case.state.value}")

# Transition 1: Start investigation
case.transition(
    CaseState.INVESTIGATING, "SOAR_AUTO",
    "begin_investigation",
    "Playbook P04_NETWORK_INTRUSION initialized. Collecting evidence."
)

# Add evidence from tools
case.add_evidence("threat_intel", {"vt_malicious": 47, "tags": ["tor", "scanner"]})
case.add_evidence("siem", {"total_events": 1247, "targets_attempted": 23})
case.add_evidence("asset_cmdb", {"hostname": "payment-processor-01", "criticality": "CRITICAL", "pci": True})

# Generate LLM summary
print(f"\n  [SUMMARY REQUESTED]")
summary = case.generate_summary()
print(f"  {summary}")
case.recommended_action = "Block src_ip at perimeter firewall. Increase monitoring on payment-processor-01."

# Transition 2: Human gate — action requires approval
case.transition(
    CaseState.PENDING_APPROVAL, "SOAR_AUTO",
    "human_gate_triggered",
    f"Recommended action: {case.recommended_action} | Human approval required."
)

print(f"\n  >>> HUMAN JUDGMENT GATE <<<")
print(f"  Alert sent to analyst via PagerDuty. Awaiting approval...")
print(f"  [Simulating analyst review and approval]")

# Simulate analyst approval
case.analyst_decision = "APPROVED"
case.transition(
    CaseState.RESPONDING, "analyst.jones",
    "analyst_approved",
    "Analyst approved block action. Proceeding with firewall rule."
)

# Transition 3: Close after response
case.add_evidence("firewall", {"action": "block_rule_applied", "rule": "SOAR-AUTO-BLOCK-185-220-101-47"})
case.transition(
    CaseState.CLOSED, "SOAR_AUTO",
    "response_confirmed",
    "Firewall rule applied. IP blocked at perimeter. Case closed."
)

# Attempt invalid transition (should fail)
print(f"\n  [Testing invalid transition: CLOSED -> INVESTIGATING]")
case.transition(CaseState.INVESTIGATING, "test", "invalid", "This should fail")

case.print_timeline()
print(f"\n  [CASE] Final state: {case.state.value} | "
      f"Events: {len(case.timeline)} | Evidence items: {len(case.evidence)}")

## Demo 5: Full SOAR Pipeline — Alert to Closure

**All pieces together** in a complete end-to-end SOAR pipeline using the
Coordinator-Worker-Delegator (CWD) model.

```
ALERT ARRIVES
     |
     v
COORDINATOR (LLM) - Plans investigation, decomposes into tasks
     |
     v
DELEGATOR - Routes tasks to appropriate workers
     |
     +----------+----------+----------+
     |          |          |          |
     v          v          v          v
NETWORK     IDENTITY    THREAT     ASSET
WORKER      WORKER      INTEL      WORKER
(parallel)  (parallel)  WORKER     (parallel)
     |          |       (parallel)    |
     +----------+----------+----------+
                     |
                     v
             COORDINATOR synthesizes
                     |
                     v
           HUMAN GATE (if needed)
                     |
                     v
          RESPONSE + CASE UPDATE + NOTIFY
```

**The graduated response principle:**

| Tier | Action | Automation Level | Example |
|------|--------|-----------------|--------|
| T1 | Observe | Fully automated | Add IP to watchlist |
| T2 | Soft control | Automated | Rate-limit at edge |
| T3 | Hard control | Human approval required | Block at firewall |
| T4 | Critical response | Always human | CISO notification |

*Never jump from alert to block without going through the tiers.
The blast radius of a wrong block action scales with asset criticality.*

In [ ]:
# ── Demo 5: Full SOAR Pipeline ─────────────────────────────────────────────────

@dataclass
class PlaybookState:
    """Shared state passed between all SOAR pipeline stages."""
    alert: Alert
    enriched: Optional[EnrichedAlert] = None
    playbook_id: str = ""
    tool_results: List[ToolResult] = field(default_factory=list)
    case: Optional[IncidentCase] = None
    coordinator_plan: str = ""
    final_decision: str = ""
    requires_human_approval: bool = False
    response_tier: int = 1
    notifications_sent: List[str] = field(default_factory=list)

def soar_coordinator(state: PlaybookState) -> PlaybookState:
    """Coordinator: plans investigation, synthesizes results, makes final decision."""
    print("\n  [COORDINATOR] Planning investigation...")

    if state.enriched and state.enriched.risk_score > 0:
        plan_prompt = (
            f"You are the SOAR coordinator. Plan this investigation:\n"
            f"Alert: {state.alert.alert_type} from {state.alert.src_ip}\n"
            f"Risk score: {state.enriched.risk_score}\n"
            f"Target asset criticality: {state.enriched.asset_info.get('criticality')}\n"
            f"PCI environment: {state.enriched.asset_info.get('handles_pci')}\n"
            f"List 3 investigation tasks for workers. Be specific. Under 80 words."
        )
        state.coordinator_plan = llm_analyze(plan_prompt, max_tokens=120)
    else:
        state.coordinator_plan = (
            "1. Network worker: verify firewall logs for this IP's activity history. "
            "2. Threat intel worker: cross-reference IP against all feeds. "
            "3. Asset worker: identify all systems potentially exposed."
        )
    print(f"  [COORDINATOR] Plan: {state.coordinator_plan[:150]}...")
    return state

def soar_delegator_and_workers(state: PlaybookState) -> PlaybookState:
    """Delegator dispatches tasks to workers and collects results in parallel."""
    print("\n  [DELEGATOR] Dispatching workers...")

    alert = state.alert

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
        futures = {
            pool.submit(call_edr_api, alert.src_ip, alert.target_host): "network_worker",
            pool.submit(call_firewall_api, alert.src_ip, "check"): "firewall_worker",
            pool.submit(call_siem_api, alert.src_ip, 30): "siem_worker",
            pool.submit(enrich_threat_intel, alert.src_ip): "threat_intel_worker",
        }
        for future in concurrent.futures.as_completed(futures):
            worker = futures[future]
            result = future.result()
            if isinstance(result, ToolResult):
                state.tool_results.append(result)
                print(f"  [{worker.upper()}] {result.tool}: {result.duration_ms}ms")
            else:
                # threat intel returns dict directly
                state.tool_results.append(
                    ToolResult("threat_intel", "lookup", True, result, duration_ms=50)
                )
                print(f"  [{worker.upper()}] threat_intel: malicious={result.get('vt_malicious',0)}")

    print(f"  [DELEGATOR] All workers complete. {len(state.tool_results)} results collected.")
    return state

def soar_decision_and_response(state: PlaybookState) -> PlaybookState:
    """Coordinator synthesizes worker results and determines response tier."""
    print("\n  [COORDINATOR] Synthesizing results and determining response...")

    risk = state.enriched.risk_score if state.enriched else 0.5
    pci = state.enriched.asset_info.get("handles_pci", False) if state.enriched else False

    # Determine graduated response tier
    if risk >= 0.85 or pci:
        state.response_tier = 3   # Hard control — human approval required
        state.requires_human_approval = True
        state.final_decision = (
            f"TIER-3 RESPONSE: Block {state.alert.src_ip} at perimeter firewall. "
            f"Increase monitoring on {state.alert.target_host}. "
            f"REQUIRES ANALYST APPROVAL (PCI system + high risk score)."
        )
    elif risk >= 0.60:
        state.response_tier = 2   # Soft control — automated
        state.requires_human_approval = False
        state.final_decision = (
            f"TIER-2 RESPONSE: Rate-limit {state.alert.src_ip} at edge. "
            f"Add to enhanced-monitoring watchlist. No analyst approval needed."
        )
    else:
        state.response_tier = 1   # Observe — automated
        state.requires_human_approval = False
        state.final_decision = (
            f"TIER-1 RESPONSE: Add {state.alert.src_ip} to watchlist. "
            f"Monitor for 24h. Auto-close if no further activity."
        )

    print(f"  [DECISION] Tier {state.response_tier}: {state.final_decision}")
    print(f"  [DECISION] Human approval required: {state.requires_human_approval}")
    return state

def soar_notify(state: PlaybookState) -> PlaybookState:
    """Send notifications based on response tier and approval requirements."""
    print("\n  [NOTIFY] Sending notifications...")

    notifications = []
    if state.requires_human_approval:
        notifications.append("PagerDuty: on-call SOC analyst (P1 incident)")
        notifications.append(f"Slack #soc-alerts: {state.case.case_id} requires approval")
        if state.enriched and state.enriched.asset_info.get("handles_pci"):
            notifications.append("Email: security-manager@corp.com (PCI system involved)")
    else:
        notifications.append(f"Slack #soc-auto: Tier {state.response_tier} action executed")

    notifications.append(f"ServiceNow: Case {state.case.case_id} updated")

    for n in notifications:
        print(f"  [SENT] -> {n}")
        state.notifications_sent.append(n)

    return state

def run_soar_pipeline(alert: Alert) -> PlaybookState:
    """Run the complete end-to-end SOAR pipeline."""
    print("=" * 65)
    print(f"FULL SOAR PIPELINE")
    print(f"Alert: {alert.alert_id} | {alert.alert_type}")
    print("=" * 65)

    state = PlaybookState(alert=alert)
    pipeline_start = time.time()

    # Stage 1: Enrich
    print("\n[STAGE 1] Alert Enrichment")
    state.enriched = enrich_alert(alert)

    # Stage 2: Coordinator plans
    print("\n[STAGE 2] Coordinator Planning")
    state = soar_coordinator(state)

    # Stage 3: Workers execute in parallel
    print("\n[STAGE 3] Worker Dispatch (parallel)")
    state = soar_delegator_and_workers(state)

    # Stage 4: Decision + graduated response
    print("\n[STAGE 4] Decision & Response")
    state = soar_decision_and_response(state)

    # Stage 5: Case management
    print("\n[STAGE 5] Case Management")
    state.case = IncidentCase(
        case_id=f"CASE-{random.randint(1000,9999)}",
        alert_id=alert.alert_id,
        title=f"{alert.alert_type} from {alert.src_ip}",
        severity="CRITICAL" if state.enriched.risk_score >= 0.8 else "HIGH"
    )
    state.case.transition(CaseState.INVESTIGATING, "SOAR_AUTO",
                          "pipeline_started", "Full SOAR pipeline initiated")
    for tr in state.tool_results[:2]:
        state.case.add_evidence(tr.tool, tr.data)

    if state.requires_human_approval:
        state.case.transition(CaseState.PENDING_APPROVAL, "SOAR_AUTO",
                              "human_gate", state.final_decision)
    else:
        state.case.transition(CaseState.RESPONDING, "SOAR_AUTO",
                              "auto_response", state.final_decision)
        state.case.transition(CaseState.CLOSED, "SOAR_AUTO",
                              "response_complete", "Tier 1/2 response applied automatically")

    # Stage 6: Notify
    print("\n[STAGE 6] Notifications")
    state = soar_notify(state)

    total_duration = time.time() - pipeline_start

    print(f"\n{'=' * 65}")
    print(f"PIPELINE COMPLETE — {total_duration*1000:.0f}ms total")
    print(f"  Case: {state.case.case_id} | State: {state.case.state.value}")
    print(f"  Response Tier: {state.response_tier} | Human Gate: {state.requires_human_approval}")
    print(f"  Notifications: {len(state.notifications_sent)}")
    print(f"  Manual equivalent: ~25 minutes -> {total_duration:.1f} seconds automated")
    print(f"{'=' * 65}")

    return state

# ── Run the full pipeline ──────────────────────────────────────────────────────
soar_alert = Alert(
    "ALT-003", "phishing", "185.220.101.47", "10.0.5.44",
    "2024-12-01T03:14:00Z",
    "Email from spoofed CFO address requesting urgent wire transfer of $240,000. "
    "Domain registered 2 days ago. CFO target handles wire transfers. No prior relationship."
)

final_state = run_soar_pipeline(soar_alert)

## Summary: What You Built

You now have a working **SOAR orchestration system** demonstrating all core patterns:

| Demo | Pattern | Key Concept |
|------|---------|------------|
| **Alert Enrichment** | Pipeline/chaining | Sequential enrichment, weighted risk scoring |
| **Playbook Router** | LLM classification | Context-aware routing beyond alert type matching |
| **Multi-Tool Orchestration** | Parallel execution | Parallel workers: time = MAX not SUM |
| **Case Management** | State machine | Valid transitions enforced, timeline immutable |
| **Full Pipeline** | CWD model | Coordinator + Workers + Graduated response |

### The Numbers That Matter

```
Manual investigation:  20-35 minutes per alert
SOAR pipeline:         90 seconds per alert
Speed improvement:     ~95% reduction in MTTR

Sequential enrichment: 400ms (sum of all step durations)
Parallel enrichment:   ~200ms (max of all step durations)
Parallelism gain:      ~2x speedup from architecture alone
```

### The Non-Negotiable Guardrail

> **Tier 3 (block) and Tier 4 (executive) actions ALWAYS require human approval.**
> The AI collects evidence, scores risk, and recommends. The analyst decides.
> Automated false-positive blocks on production systems cause outages.
> Human oversight is not optional — it is the architecture.

### Production Upgrade Path

```
Simulated enrichment APIs    ->  Real VirusTotal, MaxMind, Shodan connectors
Simulated tool APIs          ->  CrowdStrike, Palo Alto, Splunk via MCP servers
In-memory case tracking      ->  ServiceNow SOAR / Palo Alto XSOAR / Swimlane
Simulated notifications      ->  Real PagerDuty, Slack, email integration
Haiku for routing            ->  claude-sonnet-4-5-20250514 for complex decisions
Sequential playbook steps    ->  LangGraph for stateful, resumable workflows
Manual RLHF capture          ->  Structured feedback DB feeding model retraining
```

**Next: Chapter 75 — Network Anomaly Detection** — the ML models that generate
the alerts that SOAR playbooks respond to. Detection and orchestration together
close the full automated response loop.

In [ ]:
# ── Production Deployment Checklist + Key Formulas ────────────────────────────
print("CHAPTER 74: SOAR PRODUCTION DEPLOYMENT CHECKLIST")
print("=" * 65)

checklist = [
    # Architecture
    ("Architecture",   "Playbook as DAG: define nodes (actions) + edges (conditions)"),
    ("Architecture",   "State passed explicitly between stages — no shared globals"),
    ("Architecture",   "Parallel workers for independent enrichment steps (MAX not SUM)"),
    ("Architecture",   "Graduated response: Tier 1 auto, Tier 2 auto, Tier 3 human gate"),
    # Integration
    ("Integration",    "Rate-limit all SOAR-to-device calls (firewall APIs: 10 req/min)"),
    ("Integration",    "Retry with exponential backoff on transient API failures"),
    ("Integration",    "MCP servers for standardized tool integration (NETCONF analogy)"),
    ("Integration",    "Circuit breaker: stop calling failed tools, don't cascade errors"),
    # Guardrails
    ("Guardrails",     "Human gate before ALL Tier 3+ (irreversible, high blast radius)"),
    ("Guardrails",     "Immutable audit trail: every action + actor + timestamp logged"),
    ("Guardrails",     "Rollback capability for every automated network device change"),
    ("Guardrails",     "Low confidence -> human gate (never auto-act on uncertainty)"),
    # Metrics
    ("Metrics",        "Track MTTD: time from attacker action to first alert"),
    ("Metrics",        "Track MTTR: time from alert to completed response action"),
    ("Metrics",        "Track automation rate: % alerts fully resolved without analyst"),
    ("Metrics",        "Track analyst FP rate: % analyst-reviewed cases = false positive"),
    # RLHF
    ("RLHF",           "Capture analyst decisions in structured form from day 1"),
    ("RLHF",           "Monthly retrain cycle: analyst feedback -> improved triage models"),
]

current_cat = ""
for cat, item in checklist:
    if cat != current_cat:
        print(f"\n[{cat}]")
        current_cat = cat
    print(f"  + {item}")

print("\n" + "=" * 65)
print("KEY FORMULAS")
print("=" * 65)
print("Risk Score    = 0.35*(threat_intel) + 0.25*(geo_risk) + 0.25*(asset_crit) + 0.15*(history)")
print("Sequential    = SUM(step durations) | Parallel = MAX(step durations)")
print("Tier routing  = risk >= 0.85 or PCI -> Tier 3 (human gate)")
print("              = risk >= 0.60         -> Tier 2 (soft control, auto)")
print("              = risk <  0.60         -> Tier 1 (observe, auto)")
print("MTTR target   = < 120 seconds (automated) | < 10 minutes (human-reviewed)")
print("Automation %  = (alerts_auto_resolved / total_alerts) * 100  [target: 80%+]")
print("Analyst FP %  = (false_positives / analyst_reviewed) * 100   [target: <20%]")
print("\nChapter 74 complete. Orchestrate everything. Automate the routine. Reserve judgment for judgment.")